# Study 935 — Value Averaging 📐

**Edleson's rule says: don't invest the same *amount* each month — own the same
*value* each month.** Set a value path in advance, then each month buy or sell
whatever it takes to land on it. The path climbs smoothly, the market does not, so
the rule automatically buys more after a fall and sells after a rally. Its author
reports it beats dollar-cost averaging almost always.

It does — on the metric his book quotes. This study asks what happens when you also
count the **cash the rule needs to exist**: the buffer that funds the extra purchases
in falling markets, and the idle money the rule is *not* investing when the market
runs away from the path.

We test it on **SPY vs BIL** (cash) daily total-return closes, 2007-05-30 → 2026-06-30
(4,802 days), over **every** rolling 36-month savings programme
(193 of them), both arms handed identical committed capital, one execution
lag, 1 bp one-way.

*Numbers below are the frozen headline (`docs/results.md`, Fingerprint `9cce1b76d021`); the
live cells run the offline synthetic control only. As-of 2026-06-30.*


## 1. The two savers

Both put aside the same £1 a month for three years. Both start with the same reserve of **6 months' contributions** sitting in T-bills. The DCA saver buys £1 of the market every month and never thinks again. The value-averaging saver checks a chart each month: if the pot is below the line, buy the difference; if it is above, **sell** the difference and put the money back in T-bills.

At the end we simply ask who has more — counting *everything*, shares and cash.

In [1]:
R = dict(gap=-1.372, win=29.0, win_lo=23.1, win_hi=35.8, t_hac=-3.87,
         irr_eq_va=14.9, irr_eq_dca=13.97, irr_prog_va=10.07, irr_prog_dca=10.64)
print('VA minus DCA, final wealth: %+.2f cents per pound saved' % R['gap'])
print('value averaging finishes ahead in %.1f%% of the 193 programmes '
      '(95%% range %.1f-%.1f%%)' % (R['win'], R['win_lo'], R['win_hi']))

VA minus DCA, final wealth: -1.37 cents per pound saved
value averaging finishes ahead in 29.0% of the 193 programmes (95% range 23.1-35.8%)


## 2. So why does everyone say it wins?

Because of *which* return you measure. If you compute the return on the money **that actually went into shares** — the number the book quotes — value averaging really does win: **14.90%/yr vs 13.97%/yr**, a +0.92 pp lead. But that number ignores the reserve sitting in the bank *making the purchases possible*, and it ignores the cash handed back after every sale. Count the whole account and the ranking flips: **10.07%/yr vs 10.64%/yr**.

That is the entire trick. Same two savers, same tape, two different answers — because one measure quietly leaves out most of one saver's money.

> 🔬 **For the quants:** the equity-only IRR is near-invariant to the buffer size across the whole sweep (14.73% at no buffer, 14.91% at 24 months' worth), while the whole-programme IRR halves over the same range (12.97% → 6.22%). That gap between the two columns is precisely the tell that the famous number is not measuring the programme.

## 3. What is really going on: a hidden dial on how much you own

Over these nineteen years the market grew faster than the flat value path. So the value-averaging saver was constantly being told to *sell*, and spent the period with only **63.4%** of the account in shares against the DCA saver's **69.9%**. Less in the market, less of the market's return.

Tilt the value path upward and the gap moves with the equity weight, in lockstep:

| assumed path growth | equity weight | VA minus DCA |
|---|--:|--:|
| 0%/yr (the book's basic path) | 63.4% | -1.37c |
| 4%/yr | 65.4% | -0.64c |
| 8%/yr | 67.3% | +0.10c |
| 12%/yr | 69.3% | +0.83c |

The growth rate of the path is not a detail; it *is* the strategy. And nothing on the tape tells you what to set it to — you have to guess, in advance, at the return you are trying to earn.

## 4. The cash it demands, when you can least spare it

With a 6-month reserve the rule ran out of money in **6 of 193** programmes — and all six started in the summer and autumn of **2007**, i.e. straight into the financial crisis. At the worst point the rule asked for **3.3 extra months of contributions in one month** — and **6.6 months' worth in total** over the programme's binding months — at the exact moment a saver is least able to find them. Run it with no reserve at all and it is unfundable in **86%** of programmes.

The demand is not a rare tail — it is concentrated in precisely the market it is supposed to be exploiting.

## 5. Even the fair fight is not evidence of skill

Give DCA the *same* equity weight (dial its monthly purchase down to 0.90) and value averaging does finally win: **+0.70 cents**. But run the identical exercise on a **coin-flip market** — a random walk with SPY's own drift and volatility, where by construction there is nothing to predict — and the contrarian schedule earns about the same thing anyway (spread -2.84 to +1.80 cents across 12 such worlds, 2 of which beat the real tape outright). The real tape's win sits at z = +0.66 inside that noise. Buying the dips of a random walk pays a small mechanical bonus; that is all this is.

## 6. Live check — the machinery is unbiased (offline synthetic)

If prices genuinely swing back and forth, value averaging *should* clean up. The cell below plants exactly that and checks the harness finds it — then checks the harness stays silent on a tape with no swings at all. **Synthetic data, not the real tape.**

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from value_avg import data, strategy as st
planted, _ = data.synthetic_daily(n_years=12, signal_strength=1.0, seed=935)
quiet,   _ = data.synthetic_daily(n_years=12, signal_strength=0.0, seed=935, vol_ann=0.0)
pl = st.exposure_matched_race(planted['asset'], planted['cash'], 36,
                              tol=0.01, max_iter=6, buffer_mult=6.0, cost_bps=1.0)
qt = st.exposure_matched_race(quiet['asset'], quiet['cash'], 36,
                              tol=0.005, max_iter=8, buffer_mult=6.0, cost_bps=1.0)
print('SYNTHETIC swinging market : VA beats DCA by %+.2f cents (t=%+.2f) -- it works when there is something to work on'
      % (pl['gap_mean_cents'], pl['t_hac']))
print('SYNTHETIC flat market     : VA beats DCA by %+.2f cents -- nothing to harvest, nothing claimed'
      % qt['gap_mean_cents'])

SYNTHETIC swinging market : VA beats DCA by +8.18 cents (t=+4.26) -- it works when there is something to work on
SYNTHETIC flat market     : VA beats DCA by -0.11 cents -- nothing to harvest, nothing claimed


## Verdict

- **Signal — None.** Value averaging's advantage over dollar-cost averaging does not exist on this tape once you count the cash it needs: **-1.37 cents per pound saved** (HAC *t* = -3.87), ahead in only **29%** of programmes, negative in both eras and on both cross-check sleeves. The one setting where it wins is the one that quietly raises its equity weight, and the exposure-matched residual is indistinguishable from what a coin-flip market pays.
- **Tradability — Mirage.** The advertised edge is an accounting choice, not a return: the equity-only IRR says **+0.92 pp/yr** in *every* configuration we ran, including the ones where the saver ends up poorer. Costs are irrelevant (the rule trades *less* notional than DCA); what is not irrelevant is the 3.3-months-of-savings cash call it can make in a single month during a crash.